1. Setup

In [ ]:
from src.bronze.pipeline import BronzePipeline

pipeline = BronzePipeline()
spark = pipeline.spark

2. CSV

In [ ]:
csv_output_path = pipeline.run(
    source_path="/app/data/raw_local/RAW/2024-07-13_0800/CONTROLE DE MEDICOES E PAGAMENTOS/ControleMedicoesPagamentos.csv",
    source_type="csv",
    dataset_name="controle_medicoes_pagamentos",
    snapshot_date="2024-07-13_0800",
    source_file="ControleMedicoesPagamentos.csv",
    file_hash="manual_test_hash"
)

print(csv_output_path)
spark.read.parquet(csv_output_path).select("_snapshot_date", "_source_file", "_source_type").show(5)

3. XLSX

In [ ]:
xlsx_output_path = pipeline.run(
    source_path="/app/data/raw_local/RAW/2024-07-13_0800/CONTROLE DE MEDICOES EM ANDAMENTO/Exportação_bm_acompanhamento.xlsx",
    source_type="xlsx",
    dataset_name="controle_medicoes_andamento",
    snapshot_date="2024-07-13_0800",
    source_file="Exportação_bm_acompanhamento.xlsx",
    file_hash="manual_test_hash"
)

print(xlsx_output_path)
spark.read.parquet(xlsx_output_path).select("_snapshot_date", "_source_file", "_source_type").show(5)

4. XLSB / NACT

In [ ]:
xlsb_output_path = pipeline.run(
    source_path="/app/data/raw_local/RAW/2024-07-13_0800/NACT/202211_ADMIN.xlsb",
    source_type="xlsb",
    dataset_name="nact",
    snapshot_date="2024-07-13_0800",
    source_file="202211_ADMIN.xlsb",
    file_hash="manual_test_hash"
)

print(xlsb_output_path)
spark.read.parquet(xlsb_output_path).select("_snapshot_date", "_source_file", "_source_type").show(5)

5. Sumário

In [ ]:
validation_results = {
    "csv": csv_output_path,
    "xlsx": xlsx_output_path,
    "xlsb": xlsb_output_path,
}

validation_results

5. Isso libera o worker para o próximo notebook.

In [ ]:
#spark.stop()

### Batch Validation

In [ ]:
# 1. Run Bronze Batch
from src.bronze.batch import run_bronze_batch

bronze_batch_df = run_bronze_batch()

bronze_batch_df["status"].value_counts()

In [ ]:
# 2. Execution Summary
bronze_batch_df[
    [
        "execution_id",
        "source_file",
        "dataset_name",
        "source_type",
        "snapshot_date",
        "status",
        "duration_seconds",
    ]
].head(10)

In [ ]:
# 3. Failed Records
bronze_batch_df[
    bronze_batch_df["status"] == "FAILED"
][
    [
        "source_file",
        "snapshot_date",
        "dataset_name",
        "error_message",
    ]
]

In [ ]:
# Validar o log persistido:
from src.config.settings import Settings
import pandas as pd

pd.read_csv(Settings.BRONZE_EXECUTION_LOG_PATH).head(10)

### Idempotência
Ex: arquivo processado 10 vezes

Em produção isso é inaceitável.
se source_file + snapshot_date + file_hash já foi processado com SUCCESS
→ não reprocessar


In [ ]:
from src.bronze.batch import run_bronze_batch

bronze_batch_df = run_bronze_batch(force_reprocess=False)

bronze_batch_df["status"].value_counts()

### Quality Log / Reject Tracking

Objetivo: registrar problemas técnicos durante leitura
sem quebrar o batch inteiro

In [ ]:
# Validação
bronze_batch_df = run_bronze_batch(force_reprocess=True)

bronze_batch_df["status"].value_counts()

In [ ]:
from src.config.settings import Settings
import pandas as pd

quality_log_df = pd.read_csv(Settings.BRONZE_QUALITY_LOG_PATH)

quality_log_df.tail(10)